# Physical Therapy Claims Filing Agent for Anthem

This notebook automates the process of filing physical therapy claims to Anthem by:
1. Parsing PDF bills to extract relevant information
2. Automating the Anthem web portal login and navigation
3. Filling out and submitting claim forms

## Prerequisites
- Physical therapy bills in PDF format
- Anthem member credentials
- Chrome or Firefox browser installed

## Setup and Installation

In [ ]:
# Install required packages (run once)
# !pip install pdfplumber selenium webdriver-manager python-dotenv pillow

In [ ]:
import pdfplumber
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
import json
import time
from dataclasses import dataclass, asdict
import os
from getpass import getpass

# Selenium imports
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("✓ Imports successful")

## Configuration

In [ ]:
# Configuration
ANTHEM_URL = "https://www.anthem.com/login"
CLAIMS_URL = "https://www.anthem.com/claims"

# File paths
BILLS_DIR = Path("./pt_bills")  # Directory containing your PDF bills
OUTPUT_DIR = Path("./claims_output")  # Directory for extracted data and logs
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Bills directory: {BILLS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Data Models

In [ ]:
@dataclass
class ClaimInfo:
    """Structured information extracted from a bill"""
    patient_name: str = ""
    patient_dob: str = ""
    member_id: str = ""
    provider_name: str = ""
    provider_address: str = ""
    provider_phone: str = ""
    provider_tax_id: str = ""
    service_date: str = ""
    diagnosis_code: str = ""
    procedure_codes: List[str] = None
    total_amount: float = 0.0
    amount_paid: float = 0.0
    invoice_number: str = ""
    
    def __post_init__(self):
        if self.procedure_codes is None:
            self.procedure_codes = []
    
    def to_dict(self):
        return asdict(self)
    
    def is_complete(self) -> bool:
        """Check if minimum required fields are populated"""
        required = [
            self.patient_name,
            self.provider_name,
            self.service_date,
            self.total_amount > 0
        ]
        return all(required)

print("✓ Data models defined")

## PDF Parsing Functions

In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract all text from a PDF file"""
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() + "\n"
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
    return text

def extract_claim_info(pdf_path: Path) -> ClaimInfo:
    """Extract claim information from a PDF bill - optimized for P3 Athletic superbills"""
    text = extract_text_from_pdf(pdf_path)
    claim = ClaimInfo()
    
    # Patient name - appears after "Patient Information" header or in table format
    patient_match = re.search(r'Name.*?\n([A-Z][a-z]+\s+[A-Z][a-z]+)', text, re.IGNORECASE)
    if not patient_match:
        patient_match = re.search(r'Patient.*?\n.*?([A-Z][a-z]+\s+[A-Z][a-z]+)\s+\d{4}-\d{2}-\d{2}', text)
    if patient_match:
        claim.patient_name = patient_match.group(1).strip()
    
    # Date of birth - in format YYYY-MM-DD or MM/DD/YYYY
    dob_match = re.search(r'(\d{4}-\d{2}-\d{2})', text)
    if not dob_match:
        dob_match = re.search(r'(?:DOB|Date of Birth)[:\s]+(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})', text, re.IGNORECASE)
    if dob_match:
        claim.patient_dob = dob_match.group(1)
    
    # Invoice number
    invoice_match = re.search(r'Invoice\s*#\s*(\d+)', text, re.IGNORECASE)
    if invoice_match:
        claim.invoice_number = invoice_match.group(1)
    
    # Provider name - after "Provider" in Session Information
    provider_match = re.search(r'(?:Provider|Invoice #\d+)\s+([A-Z][a-z]+\s+[A-Z][a-z]+(?:\s+[A-Z]+,?\s*[A-Z]+)?)', text)
    if provider_match:
        claim.provider_name = provider_match.group(1).strip()
    
    # Clinic/Practice name - first line or specific header
    clinic_match = re.search(r'^([A-Z][A-Za-z0-9\s&]+(?:Therapy|Clinic|Practice|Health|Medical))', text, re.MULTILINE)
    if clinic_match:
        # Use clinic as provider if specific provider not found
        if not claim.provider_name:
            claim.provider_name = clinic_match.group(1).strip()
    
    # Provider address - look for street address pattern
    address_match = re.search(r'(\d+\s+[A-Z][a-zA-Z\s\.]+,\s+[A-Z]{2},?\s+\d{5})', text)
    if address_match:
        claim.provider_address = address_match.group(1).strip()
    
    # Phone number
    phone_match = re.search(r'(?:Phone|Tel)[:\s]*(\d{10}|\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4})', text, re.IGNORECASE)
    if phone_match:
        claim.provider_phone = phone_match.group(1)
    
    # EIN / Tax ID
    ein_match = re.search(r'EIN\s*#?\s*(\d{2}-\d{7})', text, re.IGNORECASE)
    if ein_match:
        claim.provider_tax_id = ein_match.group(1)
    
    # NPI - store in member_id field as additional info (or add new field)
    npi_match = re.search(r'NPI\s*#?\s*(\d+)', text, re.IGNORECASE)
    # Note: NPI is provider identifier, not stored in current ClaimInfo model
    
    # Service date - "Date of Visit" field
    service_match = re.search(r'Date of Visit.*?\n([A-Z][a-z]+\s+\d{1,2},\s+\d{4})', text, re.IGNORECASE)
    if not service_match:
        service_match = re.search(r'(?:Service Date|Date of Service|DOS)[:\s]+(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})', text, re.IGNORECASE)
    if service_match:
        claim.service_date = service_match.group(1)
    
    # Diagnosis code (ICD-10)
    diagnosis_match = re.search(r'(?:Diagnosis|ICD-?10)[:\s]+([A-Z]\d{2}\.?\d{0,2})', text, re.IGNORECASE)
    if diagnosis_match:
        claim.diagnosis_code = diagnosis_match.group(1)
    
    # Procedure codes (CPT codes) - PT codes typically start with 97
    procedure_codes = re.findall(r'\b(\d{5})\b', text)
    claim.procedure_codes = [code for code in procedure_codes if code.startswith('97')]
    
    # Total amount
    total_match = re.search(r'(?:Total|Amount Due|Balance)[\s:]+\$?([0-9,]+\.\d{2})', text, re.IGNORECASE)
    if total_match:
        claim.total_amount = float(total_match.group(1).replace(',', ''))
    
    # Amount paid
    paid_match = re.search(r'(?:Amount Paid|Payment)[\s:]+\$?([0-9,]+\.\d{2})', text, re.IGNORECASE)
    if paid_match:
        claim.amount_paid = float(paid_match.group(1).replace(',', ''))
    
    return claim

def save_claim_data(claim: ClaimInfo, output_path: Path):
    """Save extracted claim data to JSON"""
    with open(output_path, 'w') as f:
        json.dump(claim.to_dict(), f, indent=2)
    print(f"✓ Saved claim data to {output_path}")

print("✓ PDF parsing functions defined")

## Step 1: Extract Information from Bills

In [ ]:
# Create bills directory if it doesn't exist
BILLS_DIR.mkdir(exist_ok=True)

# Find all PDF files in the bills directory
pdf_files = list(BILLS_DIR.glob("*.pdf"))

if not pdf_files:
    print(f"⚠ No PDF files found in {BILLS_DIR}")
    print(f"Please add your physical therapy bills (as PDF files) to the '{BILLS_DIR}' directory.")
else:
    print(f"Found {len(pdf_files)} PDF bill(s):")
    for pdf in pdf_files:
        print(f"  - {pdf.name}")

In [ ]:
# Extract information from all bills
claims = []

for pdf_path in pdf_files:
    print(f"\nProcessing: {pdf_path.name}")
    print("-" * 50)
    
    claim = extract_claim_info(pdf_path)
    
    # Save extracted data
    output_file = OUTPUT_DIR / f"{pdf_path.stem}_extracted.json"
    save_claim_data(claim, output_file)
    
    # Display extracted information
    print(f"Patient: {claim.patient_name or '❌ Not found'}")
    print(f"Provider: {claim.provider_name or '❌ Not found'}")
    print(f"Service Date: {claim.service_date or '❌ Not found'}")
    print(f"Total Amount: ${claim.total_amount:.2f}" if claim.total_amount > 0 else "Total Amount: ❌ Not found")
    print(f"Procedure Codes: {', '.join(claim.procedure_codes) if claim.procedure_codes else '❌ Not found'}")
    
    if claim.is_complete():
        print("✓ Minimum required fields extracted")
        claims.append((pdf_path, claim))
    else:
        print("⚠ Some required fields missing - you may need to fill them manually")
        claims.append((pdf_path, claim))

print(f"\n{'='*50}")
print(f"Total claims to process: {len(claims)}")

### Review and Edit Extracted Data

Before submitting, you can review and edit the extracted claim information:

In [ ]:
# Select a claim to review/edit (change index as needed)
claim_index = 0

if claims:
    pdf_path, claim = claims[claim_index]
    print(f"Reviewing claim from: {pdf_path.name}\n")
    
    # Display current data
    for field, value in claim.to_dict().items():
        print(f"{field}: {value}")
    
    # You can manually edit fields here:
    # claim.patient_name = "John Doe"
    # claim.member_id = "ABC123456"
    # etc.
else:
    print("No claims to review")

## Web Automation Functions

In [ ]:
class AnthemClaimsAgent:
    """Agent for automating Anthem claims submission"""
    
    def __init__(self, headless: bool = False):
        self.headless = headless
        self.driver = None
        self.wait = None
    
    def start_browser(self):
        """Initialize the web driver"""
        print("Starting browser...")
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        try:
            service = Service(ChromeDriverManager().install())
            self.driver = webdriver.Chrome(service=service, options=options)
            self.wait = WebDriverWait(self.driver, 10)
            print("✓ Browser started")
        except Exception as e:
            print(f"❌ Error starting browser: {e}")
            raise
    
    def login(self, username: str, password: str):
        """Log in to Anthem portal"""
        print(f"\nNavigating to {ANTHEM_URL}...")
        self.driver.get(ANTHEM_URL)
        
        try:
            # Wait for and fill username
            username_field = self.wait.until(
                EC.presence_of_element_located((By.ID, "username"))
            )
            username_field.send_keys(username)
            
            # Fill password
            password_field = self.driver.find_element(By.ID, "password")
            password_field.send_keys(password)
            
            # Click login button
            login_button = self.driver.find_element(By.ID, "submitButton")
            login_button.click()
            
            # Wait for successful login (adjust selector based on actual Anthem site)
            time.sleep(3)
            print("✓ Login successful")
            return True
            
        except TimeoutException:
            print("❌ Login timeout - elements not found")
            print("Note: This is a template. You'll need to inspect Anthem's actual login page")
            print("and update the element selectors (IDs, classes, etc.)")
            return False
        except Exception as e:
            print(f"❌ Login error: {e}")
            return False
    
    def navigate_to_claims(self):
        """Navigate to the claims submission page"""
        print("\nNavigating to claims page...")
        try:
            # This will vary based on Anthem's actual site structure
            self.driver.get(CLAIMS_URL)
            time.sleep(2)
            
            # Look for "File a Claim" or "Submit Claim" link/button
            # Example selectors (update based on actual site):
            # file_claim_link = self.wait.until(
            #     EC.element_to_be_clickable((By.LINK_TEXT, "File a Claim"))
            # )
            # file_claim_link.click()
            
            print("✓ Navigated to claims page")
            return True
        except Exception as e:
            print(f"❌ Navigation error: {e}")
            return False
    
    def fill_claim_form(self, claim: ClaimInfo, pdf_path: Path):
        """Fill out the claim submission form"""
        print(f"\nFilling claim form for {pdf_path.name}...")
        
        try:
            # NOTE: These selectors are templates and need to be updated
            # based on Anthem's actual form structure
            
            # Patient information
            if claim.patient_name:
                patient_name_field = self.driver.find_element(By.NAME, "patientName")
                patient_name_field.send_keys(claim.patient_name)
            
            if claim.service_date:
                service_date_field = self.driver.find_element(By.NAME, "serviceDate")
                service_date_field.send_keys(claim.service_date)
            
            # Provider information
            if claim.provider_name:
                provider_field = self.driver.find_element(By.NAME, "providerName")
                provider_field.send_keys(claim.provider_name)
            
            # Amounts
            if claim.total_amount > 0:
                amount_field = self.driver.find_element(By.NAME, "totalAmount")
                amount_field.send_keys(str(claim.total_amount))
            
            # Upload PDF
            file_input = self.driver.find_element(By.CSS_SELECTOR, "input[type='file']")
            file_input.send_keys(str(pdf_path.absolute()))
            
            print("✓ Form filled")
            return True
            
        except NoSuchElementException as e:
            print(f"❌ Form element not found: {e}")
            print("Note: You need to inspect Anthem's actual form and update the selectors")
            return False
        except Exception as e:
            print(f"❌ Form filling error: {e}")
            return False
    
    def submit_claim(self):
        """Submit the claim form"""
        print("\nSubmitting claim...")
        try:
            # Find and click submit button
            submit_button = self.driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
            submit_button.click()
            
            # Wait for confirmation
            time.sleep(3)
            print("✓ Claim submitted")
            return True
            
        except Exception as e:
            print(f"❌ Submission error: {e}")
            return False
    
    def close(self):
        """Close the browser"""
        if self.driver:
            self.driver.quit()
            print("\n✓ Browser closed")

print("✓ Web automation class defined")

## Step 2: Automated Claims Submission

**IMPORTANT**: Before running this section:
1. You need to inspect Anthem's actual website to update the element selectors
2. Test with ONE claim first before processing multiple claims
3. Keep your credentials secure (use environment variables or .env file)

In [ ]:
# Get credentials securely
print("Enter your Anthem credentials:")
anthem_username = input("Username: ")
anthem_password = getpass("Password: ")

In [ ]:
# Initialize the agent
agent = AnthemClaimsAgent(headless=False)  # Set to True to run without GUI

try:
    # Start browser
    agent.start_browser()
    
    # Login
    if not agent.login(anthem_username, anthem_password):
        print("\n⚠ Login failed. Please update the login selectors in the code.")
        print("Tip: Use your browser's DevTools to inspect the login form elements.")
    else:
        # Navigate to claims
        if agent.navigate_to_claims():
            # Process each claim
            for i, (pdf_path, claim) in enumerate(claims, 1):
                print(f"\n{'='*50}")
                print(f"Processing claim {i}/{len(claims)}: {pdf_path.name}")
                print(f"{'='*50}")
                
                # Fill and submit
                if agent.fill_claim_form(claim, pdf_path):
                    # Uncomment the next line when ready to actually submit
                    # agent.submit_claim()
                    print("⚠ Submission is disabled for safety. Uncomment the line above to enable.")
                    
                    # Wait before next claim
                    if i < len(claims):
                        print("\nWaiting 5 seconds before next claim...")
                        time.sleep(5)
                else:
                    print(f"⚠ Skipping submission for {pdf_path.name} due to errors")
    
    # Keep browser open for inspection
    print("\n" + "="*50)
    print("Process complete. Browser will stay open for inspection.")
    print("Close this cell's output to close the browser.")
    input("Press Enter to close browser...")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
finally:
    agent.close()

## Manual Verification and Troubleshooting

If automated submission doesn't work:
1. Use the extracted data from the JSON files in `claims_output/`
2. Manually log into Anthem and fill the form
3. Update the selectors in the `AnthemClaimsAgent` class based on what you find

### Inspecting Web Elements

To find the correct selectors:
1. Open Chrome and go to Anthem's website
2. Right-click on a form field → "Inspect"
3. Look for `id`, `name`, or `class` attributes
4. Update the selectors in the code above

## Utilities

In [ ]:
# View all extracted claims data
def view_all_claims():
    """Display all extracted claims in a readable format"""
    json_files = list(OUTPUT_DIR.glob("*_extracted.json"))
    
    if not json_files:
        print("No extracted claims found")
        return
    
    for json_file in json_files:
        print(f"\n{'='*50}")
        print(f"File: {json_file.name}")
        print(f"{'='*50}")
        
        with open(json_file) as f:
            data = json.load(f)
        
        for key, value in data.items():
            if value:  # Only show non-empty fields
                print(f"{key:20s}: {value}")

# Uncomment to view:
# view_all_claims()

## Next Steps

1. **Customize the PDF parser**: The regex patterns may need adjustment based on your specific bill format
2. **Update web selectors**: Inspect Anthem's website and update all element selectors in `AnthemClaimsAgent`
3. **Test thoroughly**: Start with one claim and verify everything works before batch processing
4. **Handle edge cases**: Add error handling for different bill formats or website changes
5. **Security**: Use environment variables or a `.env` file for credentials instead of hardcoding

### Recommended Improvements

- Add OCR support for scanned/image-based PDFs using `pytesseract`
- Implement retry logic for network failures
- Add logging to track submission history
- Create a configuration file for different insurance providers
- Add email notifications for submission status